# Exercise 7

This exercise is based on Chapter 6 (Global Spatial Autocorrelation) of the Geographic Data Science book.

The material can be found in: `GSP538/gds_book/notebooks/06_spatial_autocorrelation.ipynb`

#### Notes on Textbook

- The authors discuss a "row standardized" spatial weights matrix a few times in the chapter, but this concept is not defined. Let's take the example of rook spatial weights for the 3x3 grid that started Chapter 4. The dense matrix representation is:

&nbsp;|0|1|2|3|4|5|6|7|8|
-|-|-|-|-|-|-|-|-|-|
__0__|0|1|0|1|0|0|0|0|0|
__1__|1|0|1|0|1|0|0|0|0|
__2__|0|1|0|0|0|1|0|0|0|
__3__|1|0|0|0|1|0|1|0|0|
__4__|0|1|0|1|0|1|0|1|0|
__5__|0|0|1|0|1|0|0|0|1|
__6__|0|0|0|1|0|0|0|1|0|
__7__|0|0|0|0|1|0|1|0|1|
__8__|0|0|0|0|0|1|0|1|0|

Row standardization means summing the values in each row, and then dividing the row by that sum as shown in the row standardized weights matrix below. The first row in the matrix above has two `1`s; these sum to `2`; after dividing the row by `2`, these `1`s become `0.50` in the matrix below. This process is repeated for each row.

&nbsp;|0|1|2|3|4|5|6|7|8|
-|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
__0__|0|0.50|0|0.50|0|0|0|0|0|
__1__|0.33|0|0.33|0|0.33|0|0|0|0|
__2__|0|0.50|0|0|0|0.50|0|0|0|
__3__|0.33|0|0|0|0.33|0|0.33|0|0|
__4__|0|0.25|0|0.25|0|0.25|0|0.25|0|
__5__|0|0|0.33|0|0.33|0|0|0|0.33|
__6__|0|0|0|0.50|0|0|0|0.50|0|
__7__|0|0|0|0|0.33|0|0.33|0|0.33|
__8__|0|0|0|0|0|0.50|0|0.50|0|


- This map of the United Kingdom might help when reading the chapter.

<img src="https://www.cgpplus.co.uk/getmedia/a8cc612f-28dd-45c7-9538-a2bf48558fbf/g2wae44-cover" width=600 height=600 />


#### Answer the following written questions

There is a blank Markdown cell after each question for your answer (double click in the blank cell to type your answer). Be sure to run your Markdown cells to format your answers.

1. Classic ideas of correlation were introduced earlier in the semester using Pearson's correlation coefficient. This chapter introduces spatial autocorrelation using Moran's I (and other measures). What is needed to compute classic correlation? What is needed to compute spatial autocorrelation? What is each measuring?

2. Explain in your own words what a spatial lag is, why it can be characterized as a smoother and why it is so important for computing spatial autocorrelation measures. (Hint: be sure you have read to the end of the chapter before answering this.)

3. The second equation under the "Spatial Lag" heading is:
    $$
    y_{sl-i} = \sum_j w_{ij} y_j
    $$
    
    - Use pen-and-paper to compute the value of $y_{sl-i}$ when $y_j$ contains the values $[9,3,4,2,8,6,5]$ and $w_{ij}$ contains $[0,0,1,1,1,1,0]$ (i.e., the binary weights case). Report the result.
    - Do this again for the same $y_j$ values, but now when $w_{ij}$ contains $[0,0,0.25,0.25,0.25,0.25,0]$ (i.e., the row standardized case). Report the result.
    - Compute the average of $4$, $2$, $8$, $6$. Report the result.
    - Explain why $y_{sl-i}$ is characterized as an "average" when $w_{ij}$ comes from a row standardized weights matrix. 
    

4. What is the difference between the binary values that are needed for the join count statistic and a binary weights matrix?

5. What is the difference between the data needed to compute the join count statistic versus what is needed for the Moran's I statistic? What is the same?

6. When the `esda` package conducts the statistical test for the join count statistic or Moran's I statistic, what is the null hypothesis? How does the package derive simulated values of the null hypothesis? How does this relate to the strategies for generating a distribution from the data science book at the beginning of the course?

7. What are some differences between global Moran's I and global Getis and Ord's G? The textbook gives some description. You might find this page helpful too. https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-statistics/h-how-high-low-clustering-getis-ord-general-g-spat.htm

#### The following questions require you to run Python code.

The cell below imports GeoPandas and other packages, and the standard extra stuff to make things run smoother. Run this cell. 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import geopandas as gpd
from shapely import Polygon

8. You will be working with precinct level 2020 presidential election data for Arizona.

   > The Upshot scraped and standardized precinct-level election results from around the country, and joined this tabular data to precinct GIS data to create a nationwide election map.
   
   https://www.kaggle.com/datasets/paultimothymooney/presidential-precinct-map-2020-election-results

    Run the cell below.
    - Explain each of the five lines
    - Despite Line 3 using a rectangle as an input, the resulting data presented in the map is not a rectangle. Why is this? (Note: this is a more of a GSP 531 question.)

    **If a map is not produced, click "Help" > "Toggle Developer Tools"; then close the panel the opens on the right.**

In [ ]:
votes = gpd.read_file('data/az_precincts_2020.geojson')  #L1
maricopa = votes.loc[votes.cnty_name=='Maricopa',]       #L2
metro_phx = maricopa.loc[maricopa.within(Polygon([(-111.5,33.1),
                                                  (-112.755,33.1),
                                                  (-112.755,33.95),
                                                  (-111.5,33.95)])),]  #L3
metro_phx = metro_phx.to_crs(2223)  #L4
metro_phx.explore()  #L5

9. Voting percents.
    - Add a column to `metro_phx` with the percent of the vote that went to the Democrat (Note: Multiply by 100 so the data looks like 24.3 not 0.243.)
    - Print out the first five rows of the GeoDataFrame after adding the column
    - Based on a visual inspection of the first five rows, which precinct had the highest percent and what was it?

10. The following cell produces a map of the election results for percent Democratic votes. 
    - Run the cell. (Note: you need to match the column name to what you used in the previous question.)
    - Explain why the particular values for `vmin` and `vmax` were chosen and their effect on the legend and resulting map. (Note: this is not asking you to define `vmin` and `vmax`, but the values assigned and their effect on the cartography of the map. Hint: inspect the range of values for percent Democratic votes before answering.)
    - Does the resulting map generally appear to be a case of positive spatial autocorrelation, negative spatial autocorrelation or neither? Explain.  

In [ ]:
ax = metro_phx.plot(column='pct_dem', cmap='RdBu', legend=True, 
                    vmin=10, vmax=90, figsize=(10,6),
                    edgecolor="white", linewidth=0.2)
ax.set_axis_off();

11. Investigate the spatial lag.
    - Create a queen spatial weights matrix for `metro_phx`
    - Row standardize the weights matrix
    - Use that weights matrix to add a column to the `metro_phx` GeoDataFrame that contains the spatial lag for percent votes for the Democrat
    - Print the first few rows of `metro_phx` showing __just__ percent vote for Democrat and the lag
    - Pick two rows from the output and interpret the percent vote and the spatial lag. If the values are similar, what does this mean; what if they are dissimilar? How does this relate to spatial autocorrelation?

12. Make a map of the lagged values similar to the one from the earlier question.

    - Discuss the percent Democrat vote map relative to the lag map. What is similar? What is different. Why?

13. Print the `describe` method for percent votes Democrat and for the lag.

    - Discuss the similarities and differences between the two.
    - Why are the min/max values so different between percent votes data and the lagged data?

14. Moran's I
   - Compute Moran's I for percent Democratic votes
   - Print the Moran's I statistic and p-value
   - Explain the null hypothesis
   - Can you reject the null hypothesis at the 0.01 level?
   - Is this positive or negative spatial autocorrelation?

15. Repeat the previous question for Graham County.

16. Moran plots.
   - Run `plot_moran` on Graham County
   - Run `plot_moran` on `metro_phx`
   - Explain generally what the scatterplot demonstrates.
   - Explain generally what the reference distribution demonstrates.
   - Describe how the plots differ for the two regions.

17. Run the following cell. It reads in the fires dataset, subsets it to California and two fire cause types and presents some preliminary output. Be sure you understand everything in the cell. The resulting map presents the origin point of each fire, classed by how the fire started. This is in effect a binary variable since there are two possible causes.

    Generally, does fire cause appear to be a case of positive spatial autocorrelation, negative spatial autocorrelation or neither? Explain.

In [ ]:
# Read in fires data
fires = pd.read_csv('data/fpa_az_ca_fires.csv')
# Convert to GeoDataFrame
pt_geoms = gpd.points_from_xy(x=fires["LONGITUDE"],
                              y=fires["LATITUDE"],
                              crs="EPSG:4326")
fires = gpd.GeoDataFrame(fires, geometry=pt_geoms)
# Subset to just two fire cause types
fires = fires.loc[fires.STAT_CAUSE_DESCR.isin(['Lightning',
                                               'Arson']),]
# Subset to California
fires = fires.loc[fires.STATE=='CA',]
# Reproject to California Albers
fires = fires.to_crs(3310)
# Print out fire cause counts
print(fires.STAT_CAUSE_DESCR.value_counts())
# Create map
fires.plot(column='STAT_CAUSE_DESCR', categorical=True,
           cmap='Spectral', markersize=4, legend=True,
           figsize=(6,6));

18. Join count statistic.
    - Create a KNN spatial weights matrix for `fires` with 8 nearest neighbors
    - Add a binary column to `fires` indicating if the cause was Lightning (1) or Arson (0)
    - Compute the join count statistic for the binary column you created and using the weights matrix you created
    - Print the join count statistic value and p-value for the bb and bw cases
    - The null hypothesis for each case is spatial randomness. What does it mean to reject the null hypothesis in the bb case? What about the bw case?
    - Can you reject the bb case for this dataset? What about the bw case? What does this imply for this dataset?

19. Run the Getis-Ord G separately for fires caused by lightning and those caused by arson. For each case:
    - Create a distance band spatial weights matrix set to 100 km
    - Compute G for `FIRE_SIZE`
    - Print the G statistic and associated p-value

20. Interpret the results from the previous question. For each fire type:
    - Explain the null hypothesis and whether or not you can reject it at the 0.05 level.
    - Is the G stat positive or negative? What does this mean about the spatial pattern of fire sizes?